In [ ]:
from phidl import Device, Layer, LayerSet, Port
from phidl.device_layout import DeviceReference
from phidl import quickplot as qp
from phidl import set_quickplot_options

import phidl.geometry as pg
import phidl.utilities as pu
import phidl.routing as pr
import phidl.path as pp

import numpy as np
import pickle

from dataclasses import dataclass, field, replace

import copy

from typing import Tuple, Optional, Union

import itertools
import importlib

from components import utils
importlib.reload(utils)

from components import qrcsj_device
importlib.reload(qrcsj_device)

from components import default_layerset
importlib.reload(default_layerset)
from components import frame
importlib.reload(frame)
from components import feedline
importlib.reload(feedline)
from components import spiral
importlib.reload(spiral)
from components import junction
importlib.reload(junction)
from components import resistor
importlib.reload(resistor)
from components import junction_resistor
importlib.reload(junction_resistor)
from components import ground_capacitor
importlib.reload(ground_capacitor)
from components import squid_resistor
importlib.reload(squid_resistor)
from components import junction_squid_resistor
importlib.reload(junction_squid_resistor)
from components import squid
importlib.reload(squid)

from components.qrcsj_device import QRCSJDevice
from components.default_layerset import default_ls
from components.frame import Frame, FrameParams
from components.feedline import Feedline, FeedlineParams, SquarePortParams
from components.spiral import Spiral, SpiralParams
from components.junction import JJ, JJParams
from components.resistor import Resistor, ResParams
from components.junction_resistor import JJResistor, CapaParams
from components.ground_capacitor import GroundCapa, GroundCapaParams
from components.squid_resistor import SquidResistor, SquidParams
from components.junction_squid_resistor import JJSquidResistor
from components.squid import Squid
from components.utils import WritefieldParams

import sys
sys.path.insert(0, '../') # add the parent directory of components to sys.path

import importlib

from components import spiral
importlib.reload(spiral)
from components.spiral import Spiral, SpiralParams

from phidl import set_quickplot_options

from phidl import quickplot as qp

In [ ]:
from components.spiral import Spiral, SpiralParams

spiral = Spiral()

# number of spiral turns is a float, lengths in microns
spiral_params=SpiralParams(N=16.28, connector_shift=40)

spiral.generate_spiral(spiral_params)

In [ ]:
from components.resistor import Resistor, ResParams
from components.squid import Squid, SquidParams
from components.squid_resistor import SquidResistor, CapaParams, WritefieldParams

squid_resistor = SquidResistor()

# number of resistive segments in series is an integer, lengths in microns
res_params = ResParams(num_segments=10, segment_length=20, spacing=5, connector_width=2)
squid_params = SquidParams(bridge_width=0.4, loop_height=15)
capa_params = CapaParams(length_x=20, length_y=8)
writefield_params = WritefieldParams()

squid_resistor.generate_squid_resistor(res_params, squid_params, capa_params, writefield_params)
squid_resistor.device.mirror()

In [ ]:
def connect_to_resonator(element: Union[JJResistor, SquidResistor, JJSquidResistor, Squid, JJ],
                         element_port_name: str,
                         spiral: Spiral,
                         resonator_port_name: str,
                         capa_distance: float = 2) -> None:

    resonator = spiral.device
    element_port: Port = element.device.ports[element_port_name]

    resonator_normal = resonator.ports[resonator_port_name].normal[1] - resonator.ports[resonator_port_name].normal[0]
    element_normal = element_port.normal[1] - element_port.normal[0]

    connector_parallel = [-resonator_normal[1], resonator_normal[0]]

    angle = np.round((360/(2*np.pi)) * np.arccos(np.clip(np.dot(resonator_normal,-element_normal), -1, 1)))%360

    if angle != 0:
        element.device.rotate(-angle)
        element.device.mirror()

    element.device.move(element_port, resonator.ports[resonator_port_name])
    element.device.move((capa_distance) * resonator_normal)
    element.device.movex(origin=element_port.x, destination=resonator.ports['out'].x)
    element.device.movex(element_port.width/2)

In [ ]:
from phidl import Device

Full_device = Device()

# lengths in microns
connect_to_resonator(squid_resistor, 'capa bot', spiral, 'capa right', capa_distance=2)

Full_device << [spiral.device, squid_resistor.device]
Full_device.rotate(90)